# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step exploration of the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset is described via a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata and data
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is an object, access attributes directly

print(f"Dataset: {metadata.name}\n\nDescription: {metadata.description}\n")

## 2. Data Overview
Review available record sets (`@id`), their fields, and columns.

We print all record sets, their `@id`s, and included fields, making it easier to reference them by `@id` in downstream steps.

In [ ]:
# List all available record sets by @id and their fields

record_sets = []
if hasattr(metadata, 'recordSet'):
    # In this schema, recordSet attribute may be a list of objects or a single object
    rslist = metadata.recordSet
    if isinstance(rslist, list):
        record_sets = rslist
    elif rslist is not None:
        record_sets = [rslist]

if not record_sets:
    print("No record sets found in metadata.")
else:
    for rs in record_sets:
        print(f"RecordSet: {getattr(rs, '@id', '-')}")
        if hasattr(rs, 'field') and rs.field:
            fields = rs.field
            if not isinstance(fields, list):
                fields = [fields]
            print("  Fields:")
            for field in fields:
                field_id = getattr(field, '@id', '-')
                field_name = getattr(field, 'name', '-')
                print(f"    - {field_id}: {field_name}")
        print("")

# For further steps, collect all record set @id values
record_set_ids = [getattr(rs, '@id', None) for rs in record_sets if getattr(rs, '@id', None)]
print(f"All RecordSet @id's: {record_set_ids}")

## 3. Data Extraction
Load data from a specific record set into a pandas DataFrame for analysis.

Use the record set and field `@id` values discovered in the previous step.

In [ ]:
# Extract data from each record set referenced by @id
# For demonstration, we use the first record set available (if present)

dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

if dataframes:
    selected_rs_id = record_set_ids[0]
    print(f"Available columns for record set {selected_rs_id}:\n{dataframes[selected_rs_id].columns.tolist()}\n")
    display(dataframes[selected_rs_id].head())
else:
    print("No dataframes extracted (no record sets).")

## 4. Exploratory Data Analysis (EDA)
We'll apply common data processing steps, such as filtering, normalization, and grouping on one numeric field and one grouping field, referencing all fields by their `@id`.

Make sure to replace the `numeric_field_id` and `group_field_id` below according to column names or `@id` values observed in the DataFrame.

In [ ]:
# Choose a numeric field (column) and a grouping field by their @id
import numpy as np

rs_id = record_set_ids[0] if record_set_ids else None
if rs_id is not None:
    df = dataframes[rs_id]
    
    # Try to infer a numeric field (looks for common tokens in clinicopathological datasets)
    numeric_field_candidates = [col for col in df.columns if any(s in col.lower() for s in ['age', 'interval', 'years', 'months', 'number', 'count', 'metastasis'])]
    if numeric_field_candidates:
        numeric_field_id = numeric_field_candidates[0]
    else:
        print("No obvious numeric field found, using the entire column list:", df.columns.tolist())
        numeric_field_id = df.columns[0] if df.columns else None

    # Try to infer a grouping field (e.g., sex, msi status, anatomical site)
    group_field_candidates = [col for col in df.columns if any(s in col.lower() for s in ['sex', 'msi', 'status', 'anatomical', 'location', 'site', 'type']) and col != numeric_field_id]
    group_field_id = group_field_candidates[0] if group_field_candidates else None

    print(f"Chosen numeric field: {numeric_field_id}")
    print(f"Chosen group field: {group_field_id}")

    if numeric_field_id is not None and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        norm_col = f"{numeric_field_id}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id}:")
        display(filtered_df[[numeric_field_id, norm_col]].head())
        
        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nMean {numeric_field_id} grouped by {group_field_id}:")
            display(grouped_df.head())
    else:
        print("Selected numeric field is not actually numeric. Printing summary statistics for all columns:")
        display(df.describe(include='all'))
else:
    print("No suitable record set or dataframe to analyze.")

## 5. Visualization
Visualize the distribution of the selected numeric field and the grouping by category.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if rs_id is not None and numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), bins=12, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

    # Boxplot by group (if exists)
    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df, showmeans=True)
        plt.title(f'{numeric_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("Cannot plot distributions: suitable numeric or grouping field not found.")

## 6. Conclusion
This notebook demonstrated a reproducible workflow for exploring the FAIR² colorectal dataset using the `mlcroissant` library. Key steps included:

- Loading Croissant metadata and reviewing schema structure
- Referencing entities by their `@id` throughout
- Extracting and inspecting records as pandas DataFrames
- Conducting basic filtering, normalization, and grouped analyses
- Visualizing numeric field distributions and between-group differences

You can now extend this template to perform custom analyses by referencing new record sets, fields, and columns—always by their `@id` per best practice for FAIR, schema-driven data!